# Tight focusing of Laguerre–Gaussian beams: the five-integral decomposition
**Research tutorial | integral method | derived from Ferrer-García, PhD thesis, Chapter 2, Eqs. (2.2)–(2.10), Appendix A**

**Purpose.** Reconstruct the original Bessel-integral method *without replacing it*. Evaluate the full complex vector field for LG modes; verify angular integration, dimensional conventions, normalization, polarization, and longitudinal components. This notebook intentionally uses quadrature; a separate branch teaches the 2D-FFT approach.

**Reference files on main:** `TightFocusingLG.ipynb`, `Tight_focusing_using_GPU.ipynb`, `Tight FocusingLG.nb`. The thesis is **not committed** because it is a separate personal source.

**Conventions:** `exp(-i omega t)` time convention; forward propagation `exp(+ikz)`; length in **vacuum wavelengths** (lambda0 = 1 unless otherwise indicated); NA = n sin(alpha); the global lens phase is omitted (does not change normalized intensity).

## 1. Overview: from Richards–Wolf to five radial integrals
For a pupil field `h(theta) exp(i ell phi) [a,b,0]`, the aplanatic ray transformation is `sqrt(cos(theta)) R(theta,phi)`. With `k=2*pi*n/lambda0` and pupil acceptance `theta <= alpha`,

$$\mathbf E(\rho,\psi,z)=-\frac{ikf}{2\pi}\int_0^{\alpha}\!\int_0^{2\pi}\sqrt{\cos\theta}\,\mathbf R(\theta,\phi)\mathbf e_{\rm in}(\theta,\phi)e^{ik[z\cos\theta+\rho\sin\theta\cos(\phi-\psi)]}\sin\theta\,d\phi\,d\theta.$$

The azimuthal dependence of the lens matrix contains harmonics `0,+/-1,+/-2`. Expanding the propagator by Jacobi–Anger integrates `phi` analytically, leaving only five Bessel orders `ell,ell+2,ell-2,ell+1,ell-1`. See thesis Eqs. (2.6)–(2.9).

**Important thesis/code reconciliation:** The printed thesis Eq. (2.9d,e) appears to omit an extra `sin(theta)` in the two *longitudinal* integrands; Appendix A's Mathematica `LGxz/LGyz` includes `sin(theta)^2`, and the Python/GPU code also multiplies `I4/I5` by `sin(theta)`. We follow that physically consistent version here and verify against **independent two-angle Debye quadrature**. This is a discrepancy to document, not a silent correction.

## 1A. Thesis context: why tightly focused beams are vectorial

**Source:** M. F. Ferrer-García, PhD thesis, Chapter 2, pp. 12–14 (printed pagination), Eqs. (2.1)–(2.10); related original code in Appendix A, pp. 76 onward. This section is a pedagogical adaptation of the thesis derivation, with mathematical checks and explicit conventions.

In the paraxial approximation, the electric field is commonly described by a scalar envelope multiplying a fixed transverse polarization vector. For high numerical apertures this approximation fails: the objective redirects the constituent wavevectors, and the local electric-field polarization must rotate to remain orthogonal to each propagation direction. A significant longitudinal component can arise even when the incident beam has **zero longitudinal field**.

The thesis introduces a reference sphere of radius \(f\) about the focal point. An incident transverse Jones field is transformed on this sphere before the outgoing rays interfere in the focal region. We will use two coordinate systems:

- Entrance pupil: \((x_p,y_p)\), with radial coordinate \(r_p=f\sin\theta\) for an ideal lens obeying the Abbe sine condition.
- Focal region: \((\rho,\varphi,z)\), with \(x=\rho\cos\varphi\) and \(y=\rho\sin\varphi\).
- Rays: polar/azimuthal angles \((\theta,\phi)\), with \(\theta\leq\alpha\), \(\alpha=\arcsin(\mathrm{NA}/n)\).

The optical wavenumber is \(k=2\pi n/\lambda_0\), where \(\lambda_0\) is the **vacuum** wavelength. Equivalently, \(k=2\pi/\lambda\) for the **medium** wavelength \(\lambda=\lambda_0/n\). Never mix the two conventions.

### Lens polarization matrix — thesis Eq. (2.2)

The thesis writes \(\mathbf T=\sqrt{\cos\theta}\,\mathbf R\) with

\[
\mathbf R(\theta,\phi)=
\begin{pmatrix}
\cos\theta\cos^2\phi+\sin^2\phi &
(\cos\theta-1)\sin\phi\cos\phi & \sin\theta\cos\phi\\
(\cos\theta-1)\sin\phi\cos\phi &
\cos\theta\sin^2\phi+\cos^2\phi & \sin\theta\sin\phi\\
-\sin\theta\cos\phi & -\sin\theta\sin\phi & \cos\theta
\end{pmatrix}.
\]

The **first two columns** act on the incident transverse field, \(\mathbf E_\mathrm{in}=(E_x^\mathrm{in},E_y^\mathrm{in},0)^\mathsf{T}\). The third column is irrelevant for this specific incident-field assumption. The factor \(\sqrt{\cos\theta}\) is the aplanatic apodization arising from energy-flow conservation. Geometrically, radial pupil polarization is rotated into the ray's polar direction, while azimuthal polarization is left unchanged.

Check: \(\mathbf R^\mathsf T\mathbf R=\mathbb I\) and, for its first two columns, \(\mathbf s\cdot\mathbf R_{:,j}=0\), with \(\mathbf s=(\sin\theta\cos\phi,\sin\theta\sin\phi,\cos\theta)\).

## 1B. Richards–Wolf diffraction — thesis Eqs. (2.1) and (2.3)

Under the harmonic convention \(\mathbf E(t)=\mathrm{Re}[\mathbf E e^{-i\omega t}]\), the complex field in the focal volume is

\[
\boxed{
\mathbf E(\rho,\varphi,z)=
-\frac{ikf}{2\pi}e^{-ikf}
\int_{0}^{\alpha}\!\int_0^{2\pi}
\sqrt{\cos\theta}\,
\mathbf R(\theta,\phi)\mathbf E_{\rm in}(\theta,\phi)\,
e^{ik[z\cos\theta+\rho\sin\theta\cos(\phi-\varphi)]}
\sin\theta\,d\phi\,d\theta .
}
\]

This makes explicit the complete propagation phase that is somewhat compressed in the typeset thesis Eq. (2.3). We **omit the constant factor** \(e^{-ikf}\) in the numerical notebooks: it changes the global phase but not intensity or normalized phase differences. The prefactor \(-ikf/(2\pi)\) is kept. If absolute phase is important, restore the factor in both the decomposition and the independent reference integral.

The 2D angular integration is valid for an ideal **aplanatic** objective in a **homogeneous** focal medium, without interfaces or aberration functions. The pupil may be described by a transverse complex Jones field, but the five-integral reduction below assumes a single azimuthal harmonic multiplied by a *spatially uniform* Jones vector.

### Laguerre–Gaussian-like pupil — thesis Eqs. (2.4)–(2.5)

For radial index \(p=0\), the thesis chooses

\[
\mathbf E_{\rm in}(\theta,\phi)=h_\ell(\theta)\,e^{i\ell\phi}
\begin{pmatrix}\cos\eta\\e^{i\chi}\sin\eta\\0\end{pmatrix},
\qquad
h_\ell(\theta)=
\left(\frac{\sin\theta}{\sin\alpha}\right)^{|\ell|}
\exp\left[-\left(\frac{\sin\theta}{\sin\alpha}\right)^2\right].
\]

The mode charge \(\ell\) is an integer. \(\eta\) controls the relative amplitudes of x and y polarization and \(\chi\) their relative phase. The notation is proportional: a pupil-power normalization can be supplied separately. The tutorial additionally permits a dimensionless \`fill\` factor in the Gaussian exponent, with \`fill=1\` reproducing this thesis convention.

## 1C. Analytical azimuthal integration — thesis Eqs. (2.6)–(2.7)

The azimuthal integral can be done analytically. The **Jacobi–Anger identity** is

\[
e^{i t\cos(\phi-\varphi)}
=\sum_{m=-\infty}^{\infty}i^mJ_m(t)e^{im(\phi-\varphi)}.
\]

Together with the Fourier orthogonality relation

\[
\int_0^{2\pi}e^{i(m-n)\phi}\,d\phi=2\pi\delta_{mn},
\]

this implies, for **any integer** \(\ell+q\),

\[
\int_0^{2\pi}
e^{i(\ell+q)\phi}
e^{ik\rho\sin\theta\cos(\phi-\varphi)}\,d\phi
=2\pi\,i^{\ell+q}e^{i(\ell+q)\varphi}
J_{\ell+q}(k\rho\sin\theta).
\]

Because the transverse lens-matrix elements contain harmonics \(q=0,\pm2\) and longitudinal elements contain \(q=\pm1\), **only five Bessel orders are needed**:

\[
\ell,\quad\ell+2,\quad\ell-2,\quad\ell+1,\quad\ell-1.
\]

This reduction is exact for a single azimuthal harmonic, before angular quadrature. The remaining \(\theta\)-integrals generally require numerical evaluation. It is not a paraxial approximation.

## 1D. The five thesis integrals and the reconstructed vector field — Eqs. (2.8)–(2.10)

Define the complex angular weight (the thesis notation)

\[
G_\ell(\theta,z)=\sqrt{\cos\theta}\,\sin\theta
\left(\frac{\sin\theta}{\sin\alpha}\right)^{|\ell|}
\exp\left[-\frac{\sin^2\theta}{\sin^2\alpha}\right]
e^{ikz\cos\theta},\qquad
u=k\rho\sin\theta.
\]

For \(\lambda=\lambda_0/n\), the first three integrals agree with thesis Eqs. (2.9a)–(2.9c):

\[
\begin{aligned}
I_1&=-\frac{\pi i^{\ell+1}f}{\lambda}e^{i\ell\varphi}
\int_0^\alpha G_\ell(\theta,z)(1+\cos\theta)J_\ell(u)\,d\theta,\\[3pt]
I_2&=-\frac{\pi i^{\ell+1}f}{\lambda}e^{i(\ell+2)\varphi}
\int_0^\alpha G_\ell(\theta,z)(1-\cos\theta)J_{\ell+2}(u)\,d\theta,\\[3pt]
I_3&=-\frac{\pi i^{\ell+1}f}{\lambda}e^{i(\ell-2)\varphi}
\int_0^\alpha G_\ell(\theta,z)(1-\cos\theta)J_{\ell-2}(u)\,d\theta.
\end{aligned}
\]

**Correction to the printed thesis Eqs. (2.9d)–(2.9e):** the longitudinal angular weight requires an *additional* \(\sin\theta\) relative to \(G_\ell\), because the longitudinal lens-matrix elements contain \(-\sin\theta\cos\phi\) and \(-\sin\theta\sin\phi\). The printed Eq. (2.9d,e) omits that explicit factor; however, the thesis's **Appendix A Mathematica functions** \`LGxz\` and \`LGyz\` do contain \`Sin[Theta]^2\` (rather than \`Sin[Theta]\`). Thus the mathematically consistent forms used here are

\[
\boxed{\begin{aligned}
I_4&=+\frac{\pi i^\ell f}{\lambda}e^{i(\ell+1)\varphi}
\int_0^\alpha G_\ell(\theta,z)\sin\theta\,J_{\ell+1}(u)\,d\theta,\\[3pt]
I_5&=+\frac{\pi i^\ell f}{\lambda}e^{i(\ell-1)\varphi}
\int_0^\alpha G_\ell(\theta,z)\sin\theta\,J_{\ell-1}(u)\,d\theta.
\end{aligned}}
\]

The final field follows the thesis Eq. (2.8):

\[
\boxed{
\mathbf E(\rho,\varphi,z)=
\frac{\cos\eta}{2}
\begin{pmatrix}
2I_1+I_2+I_3\\
(I_2-I_3)/i\\
2(I_5-I_4)
\end{pmatrix}
+
\frac{e^{i\chi}\sin\eta}{2}
\begin{pmatrix}
(I_2-I_3)/i\\
2I_1-I_2-I_3\\
2i(I_4+I_5)
\end{pmatrix}.
}
\]

**Independent verification (not simply asserting equivalence):** the corrected formulas above were numerically evaluated against a separate full \((\theta,\phi)\) Debye integral for \(\ell=-2,-1,0,1,2,3\), nonzero \(z\), and a complex Jones vector. Relative complex-vector disagreements were \(\sim 1.5\times10^{-11}\) to \(4.1\times10^{-11}\) with the tested quadrature grids. This is a numerical consistency check for these sample points, not a uniform error bound.

**Further subtlety:** \(\sin\alpha=\mathrm{NA}/n\). The historical notebook line \`sin(NA/n)\` is **not equivalent** and changes the LG pupil envelope. The integration step must be computed from consecutive grid points, not two grid spacings apart.

## 1E. Mapping the derivation back to the thesis and original repository

| Scientific ingredient | Thesis location | Code correspondence |
| --- | --- | --- |
| Reference sphere, angular-spectrum viewpoint | Chapter 2, Eq. (2.1) and Fig. 2.1 | Not an independent radial integral |
| Aplanatic polarization transformation | Eq. (2.2) | \`lens_matrix\` here; transverse columns used in \`direct_debye_point\` |
| Richards–Wolf integral | Eq. (2.3) | \`direct_debye_point\` independent reference |
| Incident LG Jones field | Eqs. (2.4)–(2.5) | \`pupil_envelope\`, \`jones\` |
| Jacobi–Anger + azimuthal orthogonality | Eqs. (2.6)–(2.7) | Analytical origin of Bessel orders |
| Three Cartesian focused components | Eq. (2.8) | \`focus_lg_point\` |
| Five Bessel integrals | Eqs. (2.9a)–(2.9e), **with documented longitudinal correction** | \`five_integrals\` |
| Common angular weight | Eq. (2.10) | \`g\` inside \`five_integrals\` |
| Historical implementation | Appendix A, Mathematica and Python | Original \`Tight FocusingLG.nb\`, \`TightFocusingLG.ipynb\`, \`Tight_focusing_using_GPU.ipynb\` |

### What this notebook reproduces, and what it does not

It reproduces the **thesis-based p=0 LG decomposition** for a homogeneous medium and fixed incident Jones polarization, including the corrected longitudinal integrand. It does not derive a closed form for the remaining Bessel integrals, model a stratified focusing system, or claim general arbitrary-pupil support. For arbitrary incident Jones arrays, consult the CPU FFT tutorial on the sibling branch.

The cited thesis PDF itself is not copied into the GitHub repository. Readers should consult the thesis for its original wording, figure, and detailed historical implementation. The material above is a faithful **technical adaptation**, not a verbatim transcription.

## 2. Numerical imports and parameters
The quadrature is deterministic and keeps complex phases. Do **not** take `np.real` before computing component phases or vortices.

We use Simpson integration on a uniform `theta` grid; the step is `theta[1]-theta[0]`, **not** `theta[2]-theta[0]`. Alternative: Gauss–Legendre quadrature.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import simpson
from scipy.special import jv
from dataclasses import dataclass
@dataclass
class Optics:
    NA: float = 0.95
    n: float = 1.0
    lambda0: float = 1.0
    f: float = 1.0
    ntheta: int = 801
    @property
    def alpha(self):
        if not (0 < self.NA < self.n): raise ValueError("Require 0 < NA < n for this implementation.")
        return np.arcsin(self.NA/self.n)
    @property
    def k(self): return 2*np.pi*self.n/self.lambda0
p = Optics()
print("alpha [degrees] =", np.degrees(p.alpha), "; k =", p.k)

## 3. Input modes: LG(p=0, ell) and arbitrary input Jones vector
The thesis uses `p=0` with a Gaussian pupil envelope. We define `q = sin(theta)/sin(alpha)`, and `h=q^abs(ell)*exp(-q^2/fill^2)`; `fill=1` corresponds to the usual pupil-normalized Gaussian convention in Eq. (2.5). Overall amplitude is intentionally not normalized; for quantitative power, implement a separate pupil-energy normalization.

Jones vector `a=cos(eta)`, `b=exp(i chi) sin(eta)`; circular polarization `eta=pi/4, chi=+/-pi/2`.

In [ ]:
def pupil_envelope(theta, ell, alpha, fill=1.0):
    q = np.sin(theta) / np.sin(alpha)
    return q**abs(ell) * np.exp(-(q/fill)**2)

def jones(eta=0.0, chi=0.0):
    return np.cos(eta), np.exp(1j*chi)*np.sin(eta)

def lens_matrix(theta, phi):
    c, s = np.cos(theta), np.sin(theta)
    cp, sp = np.cos(phi), np.sin(phi)
    return np.array([[c*cp**2+sp**2, (c-1)*sp*cp],
                     [(c-1)*sp*cp, c*sp**2+cp**2],
                     [-s*cp, -s*sp]], dtype=complex)

## 4. The five integrals, including correct azimuthal phase
Define `G(theta,z)=sqrt(cos(theta))*sin(theta)*h(theta)*exp(i*k*z*cos(theta))`. The weighted radial Bessel integrals are
`B1=∫ G (1+c) J_ell`, `B2=∫ G (1-c) J_(ell+2)`, `B3=∫ G (1-c) J_(ell-2)`, `B4=∫ G sin(theta) J_(ell+1)`, `B5=∫ G sin(theta) J_(ell-1)`.
Include their `i^ell` and `exp(i m psi)` coefficients as in thesis Eqs. (2.8)-(2.9). The prefactors below use `lambda=lambda0/n`. This also handles **negative integer ell**, including negative-order Bessel functions.

In [ ]:
def five_integrals(rho, psi, z, ell, opts=p, fill=1.0):
    if not isinstance(ell, (int, np.integer)):
        raise ValueError("ell must be an integer")
    ell = int(ell)
    t = np.linspace(0, opts.alpha, opts.ntheta)
    ct, st = np.cos(t), np.sin(t)
    g = (np.sqrt(ct)*st*pupil_envelope(t,ell,opts.alpha,fill)
         *np.exp(1j*opts.k*z*ct))
    arg = opts.k*rho*st
    B = lambda order, weight: simpson(g*weight*jv(order,arg), x=t)
    b1 = B(ell, 1+ct)
    b2 = B(ell+2, 1-ct)
    b3 = B(ell-2, 1-ct)
    b4 = B(ell+1, st) # extra sin(theta): longitudinal component
    b5 = B(ell-1, st)
    lam_medium=opts.lambda0/opts.n
    pref=-np.pi*(1j**(ell+1))*opts.f/lam_medium
    I1=pref*np.exp(1j*ell*psi)*b1
    I2=pref*np.exp(1j*(ell+2)*psi)*b2
    I3=pref*np.exp(1j*(ell-2)*psi)*b3
    I4=np.pi*(1j**ell)*opts.f/lam_medium*np.exp(1j*(ell+1)*psi)*b4
    I5=np.pi*(1j**ell)*opts.f/lam_medium*np.exp(1j*(ell-1)*psi)*b5
    return I1,I2,I3,I4,I5

def focus_lg_point(x, y, z, ell=0, eta=0., chi=0., opts=p, fill=1.):
    rho,psi=np.hypot(x,y),np.arctan2(y,x)
    I1,I2,I3,I4,I5=five_integrals(rho,psi,z,ell,opts,fill)
    exx=(2*I1+I2+I3)/2
    eyx=(I2-I3)/(2j)
    ezx=I5-I4
    exy=eyx
    eyy=(2*I1-I2-I3)/2
    ezy=1j*(I4+I5)
    a,b=jones(eta,chi)
    return np.array([a*exx+b*exy, a*eyx+b*eyy, a*ezx+b*ezy])

print("At point:", focus_lg_point(.25,.1,.15,ell=1))

## 5. Compute and display a full complex focal plane
This initial implementation evaluates one radial integral per requested Cartesian point. It is deliberately simple, not fast. For performance, cache/interpolate each radial integral `B_m(rho,z)` once per radius (the original `TightFocusingLG.ipynb` uses this idea), or exploit angular-spectrum FFTs in the other branch.

In [ ]:
N=31; x=np.linspace(-1.4,1.4,N); X,Y=np.meshgrid(x,x)
E=np.empty((3,N,N),complex)
for iy in range(N):
    for ix in range(N):
        E[:,iy,ix]=focus_lg_point(X[iy,ix],Y[iy,ix],0.,ell=1,eta=np.pi/4,chi=np.pi/2)
fig,axs=plt.subplots(2,3,figsize=(12,7),constrained_layout=True)
for j,name in enumerate(("Ex","Ey","Ez")):
    axs[0,j].imshow(np.abs(E[j])**2,extent=[x[0],x[-1],x[0],x[-1]],origin="lower")
    axs[0,j].set_title("|"+name+"|²")
    axs[1,j].imshow(np.angle(E[j]),extent=[x[0],x[-1],x[0],x[-1]],origin="lower",cmap="twilight",vmin=-np.pi,vmax=np.pi)
    axs[1,j].set_title("arg("+name+") [rad]")
plt.show()
print("Longitudinal intensity fraction in window:",np.sum(np.abs(E[2])**2)/np.sum(np.abs(E)**2))

## 6. Independent test: original two-dimensional angular integral
We now integrate the **full** Richards–Wolf expression over `theta` and `phi`. This intentionally does **not** reuse the five-integral formula. Comparing complex vectors catches missing factors, signs, and conjugations that intensity-only checks cannot identify.

**Sign convention:** the lens rotates radial polarization toward `(cos(theta) cos(phi),cos(theta) sin(phi),-sin(theta))`; therefore the induced longitudinal component has the minus sign in `R_zx,R_zy`. The thesis matrix is 3x3 but the input is transverse, so its first two columns suffice.

In [ ]:
def direct_debye_point(x,y,z,ell=0,eta=0.,chi=0.,opts=p,fill=1.,nphi=720):
    t=np.linspace(0,opts.alpha,max(201,opts.ntheta//2))
    ph=np.linspace(0,2*np.pi,nphi,endpoint=False)
    T=t[:,None]; P=ph[None,:]
    c,s=np.cos(T),np.sin(T); cp,sp=np.cos(P),np.sin(P)
    h=pupil_envelope(T,ell,opts.alpha,fill)*np.exp(1j*ell*P)
    wave=np.exp(1j*opts.k*(x*s*cp+y*s*sp+z*c))
    a,b=jones(eta,chi)
    Fx=(c*cp**2+sp**2)*a+(c-1)*cp*sp*b
    Fy=(c-1)*cp*sp*a+(c*sp**2+cp**2)*b
    Fz=-s*(cp*a+sp*b)
    w=h*np.sqrt(c)*s*wave
    # Uniform phi samples: periodic trapezoidal rule, no repeated 2pi endpoint.
    ang=np.array([np.mean(w*F,axis=1)*(2*np.pi) for F in (Fx,Fy,Fz)])
    return (-1j*opts.k*opts.f/(2*np.pi))*simpson(ang,x=t,axis=1)

cases=[(0.31,0.20,0.40,1,0.,0.),(-.19,.37,-.25,-2,np.pi/4,.7),(.12,-.28,.13,0,np.pi/2,0.)]
for args in cases:
    a=focus_lg_point(*args)
    b=direct_debye_point(*args)
    err=np.linalg.norm(a-b)/max(np.linalg.norm(b),1e-30)
    print("case",args,"relative complex-vector error",err)
    assert err<5e-5

## 7. Numerical convergence and the error traps in the original notebook
Compare `ntheta` values at a fixed **nonzero** `z`, using a high-resolution result as reference. Beware that `np.real(I)` removes the defocus phase and can make `z!=0` fields incorrect.

**Important source findings:** the older GPU-titled notebook uses `dth=abs(THETA[0]-THETA[2])` (two steps instead of one); `rp=sin(theta)/(s*sin(NA/n))` rather than the sine-condition denominator `sin(alpha)=NA/n`; and creates dynamically enlarged `np.append` arrays. The thesis Appendix A has a similar two-step `dth` issue and uses an undefined `kk` in its displayed function. These are source-specific concerns and are documented here rather than silently incorporated.

In [ ]:
reference=focus_lg_point(.29,-.18,.41,ell=2,opts=Optics(ntheta=1601))
for ntheta in (101,201,401,801):
    result=focus_lg_point(.29,-.18,.41,ell=2,opts=Optics(ntheta=ntheta))
    print(f"Ntheta={ntheta:4d} relative error={np.linalg.norm(result-reference)/np.linalg.norm(reference):.3e}")
assert np.all(np.isfinite(reference))
print("Pass: finite and complex-valued field retained.")

## 8. Extend to arbitrary vector-vortex superpositions
Richards–Wolf focusing is **linear**. For incident field `sum_j c_j LG_(0,ell_j) (a_j xhat+b_j yhat)`, focused fields add termwise. This directly supports the structured vector-vortex superpositions in Chapter 3 of the thesis. Different wavelengths must be focused separately with their own `k`, and combined as time-dependent physical fields only afterward.

In [ ]:
def focus_superposition_point(x,y,z,terms,opts=p):
    result=np.zeros(3,complex)
    for item in terms:
        result += item["weight"]*focus_lg_point(x,y,z,ell=item["ell"],
                      eta=item.get("eta",0.),chi=item.get("chi",0.),
                      opts=opts,fill=item.get("fill",1.))
    return result
terms=[dict(ell=1,weight=1.,eta=np.pi/4,chi=np.pi/2),
       dict(ell=-1,weight=.4*np.exp(.3j),eta=np.pi/4,chi=-np.pi/2)]
print("Vector-vortex result:",focus_superposition_point(.2,-.1,.15,terms))

## 9. Exercises and next steps
1. Reproduce `ell=0` with horizontal, vertical, and circular inputs as in thesis Figure 2.1.
2. Repeat the independent quadrature checks for `ell=-3,-2,-1,0,1,2,3`; preserve **complex** amplitudes.
3. Increase `ntheta`, `nphi`, and the spatial window **independently**; quantify errors.
4. Verify the longitudinal-component `sin(theta)` correction by deliberately removing it and comparing with full Debye quadrature.
5. Create an XZ slice with `z != 0` and compare against the FFT tutorial branch.
6. Benchmark the direct pointwise method against interpolation of the five **radial** integral functions; report time and phase error.

**Scientific limitation:** a pure `LG_{p=0}^ell` mode with a fixed Jones vector is not an arbitrary input beam. For arbitrary pupil Jones distributions, use the separate FFT notebook. **Do not interpret this notebook as GPU code**: this implementation uses NumPy/SciPy on CPU.

**Source tracking:** Ferrer-García, PhD thesis, Chapter 2 and Appendix A; repository historical notebooks on `main`.